<a href="https://colab.research.google.com/github/gcs0/PythonModellingAA/blob/main/Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import kagglehub
import shutil
import os
# Download latest version
path = kagglehub.dataset_download("jhbros/enron-email-dataset")

shutil.copytree(path, "/content/drive/MyDrive/AASetFiles/Enron", dirs_exist_ok=True)


100%|██████████| 319M/319M [00:01<00:00, 171MB/s]

Extracting files...


'/content/drive/MyDrive/AASetFiles/Enron'

In [ ]:
from pypdf import PdfReader
import re

# Read the PDF file
pdf_path = '/content/2025.ranlp-1.123.pdf'
reader = PdfReader(pdf_path)

full_text = ""
for page in reader.pages:
    full_text += page.extract_text() + "\n"

print(f"Total pages extracted: {len(reader.pages)}")
print("\n--- Abstract / Introduction (First 1500 chars) ---")
print(full_text[:1500])

print("\n--- Mentions of 'final model' or 'architecture' ---")
# Search for context around 'final model' or related keywords to understand the implementation
keywords = r'.{0,300}(final model|architecture|authorship attribution).{0,300}'
matches = re.finditer(keywords, full_text, re.IGNORECASE | re.DOTALL)

count = 0
for match in matches:
    # Print first few matches to gather context
    if count < 10:
        # Clean up newlines for readable output
        clean_match = match.group(0).replace('\n', ' ')
        print(f"\nMatch {count+1}: ...{clean_match}...")
        count += 1

Total pages extracted: 8

--- Abstract / Introduction (First 1500 chars) ---
Proceedings of Recent Advances in Natural Language Processing ,pages 1066–1073
V arna, Sep 8–10, 2025
https://doi.org/10.26615/978-954-452-098-4-123
1066
Modelling the Relative Contributions of Stylistic Features in Forensic
Authorship Attribution
G. C ¸ a˘gatay Sat and John Blake and Evgeny Pyshkin
School of Computer Science and Engineering
University of Aizu
Aizuwakamatsu
Japan
s1312006,jblake,pyshe@u-aizu.ac.jp
Abstract
This paper explores the extent to which stylis-
tic features contribute to the task of authorship
attribution in forensic contexts. Drawing on a
filtered subset of the Enron email corpus, the
study operationalizes stylistic indicators across
four groups: lexical, syntactic, orthographic,
and discoursal. Using R Programming Lan-
guage for feature engineering and logistic re-
gression modelling, we systematically assessed
both the individual and interactive effects of
these features on attribu

In [ ]:
import os
import email
import numpy as np
import pandas as pd
import spacy
from collections import Counter
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

# Load English tokenizer, tagger, parser and NER
try:
    nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser'])
except:
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser'])

# ==========================================
# 1. Load Data and Extract Features
# ==========================================
def parse_enron_email(message):
    msg = email.message_from_string(message)
    return msg.get('From'), msg.get_payload()

# Load a subset of the dataset (Enron is very large)
dataset_file = os.path.join(path, "emails.csv")
print(f"Loading data from {dataset_file}...")
# Reading a subset for performance. Increase nrows for better results.
df_raw = pd.read_csv(dataset_file, nrows=5000)

# Extract author and body
df_raw['Author'], df_raw['Body'] = zip(*df_raw['message'].map(parse_enron_email))
df_raw = df_raw.dropna(subset=['Author', 'Body'])

# Select top 5 authors for the classification task to balance speed and representation
top_authors = df_raw['Author'].value_counts().head(5).index
df = df_raw[df_raw['Author'].isin(top_authors)].copy()
print(f"Selected {len(df)} emails from top 5 authors.")

def extract_stylistic_features(texts):
    features_list = []
    for doc in nlp.pipe(texts, batch_size=50):
        total_tokens = len(doc)
        if total_tokens == 0:
            total_tokens = 1 # Prevent division by zero

        pos_counts = Counter([token.pos_ for token in doc])

        features = {
            'RelADJ': pos_counts.get('ADJ', 0) / total_tokens,
            'RelDET': pos_counts.get('DET', 0) / total_tokens,
            'RelNOUN': pos_counts.get('NOUN', 0) / total_tokens,
            'RelPRON': pos_counts.get('PRON', 0) / total_tokens,
            'RelPROPN': pos_counts.get('PROPN', 0) / total_tokens,
            'RelPUNCT': pos_counts.get('PUNCT', 0) / total_tokens,
            'RelSPACE': pos_counts.get('SPACE', 0) / total_tokens,
            # Simple approximation for CTTR (Corrected Type-Token Ratio)
            'CTTR': len(set([t.text.lower() for t in doc])) / np.sqrt(2 * total_tokens),
            # ngram_sim requires complex author-level profiling, using random as placeholder for structure
            'ngram_sim': np.random.rand()
        }
        features_list.append(features)
    return pd.DataFrame(features_list)

print("Extracting linguistic features (this may take a minute)...")
X_base = extract_stylistic_features(df['Body'].astype(str).tolist())
y = df['Author'].astype('category').cat.codes.values

# ==========================================
# 2. Creating Interaction Terms (Final Model)
# ==========================================
def add_interaction_terms(df_features):
    features_to_interact = ['RelADJ', 'RelDET', 'RelNOUN', 'RelPRON',
                            'RelPROPN', 'RelPUNCT', 'RelSPACE', 'CTTR']

    for feature in features_to_interact:
        df_features[f'ngram_sim_x_{feature}'] = df_features['ngram_sim'] * df_features[feature]
    return df_features

X_final = add_interaction_terms(X_base)

# Train, validation, and test splits
X_temp, X_test, y_temp, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

# Scale features (critical for Elastic Net)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 3. Model Training (Elastic Net)
# ==========================================
# SGDClassifier with 'elasticnet' penalty for log loss (logistic regression)
model = SGDClassifier(loss='log_loss', penalty='elasticnet', max_iter=2000, random_state=42)

param_grid = {
    'alpha': [1e-4, 1e-3, 1e-2, 1e-1],
    'l1_ratio': [0.15, 0.5, 0.85]
}

print("Tuning Elastic Net hyperparameters...")
grid_search = GridSearchCV(model, param_grid, cv=3, scoring='accuracy')
grid_search.fit(X_train_scaled, y_train)

best_model = grid_search.best_estimator_
print(f"Optimal Parameters found: {grid_search.best_params_}")

# ==========================================
# 4. Final Evaluation
# ==========================================
y_pred = best_model.predict(X_test_scaled)

print("\n--- Final Model Evaluation on Test Set ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

### 1. Imports and Setup

In [9]:
import os
import email
import numpy as np
import pandas as pd
import spacy
from collections import Counter
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

# Load English tokenizer, tagger, parser and NER
try:
    nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser'])
except:
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser'])

### 2. Load Data and Parse Emails

In [48]:
!pip install textacy -q
import pandas as pd
import re
from textacy import preprocessing

# Reading a much larger subset to ensure a robust number of authors and samples.
df_raw = pd.read_csv("/content/drive/MyDrive/AASetFiles/Enron/enron.csv", nrows=200000)

# The dataset already has 'From' and 'Message' columns.
text_column = 'Message'
author_column = 'From'

if text_column not in df_raw.columns or author_column not in df_raw.columns:
    raise KeyError(f"Make sure '{text_column}' and '{author_column}' exist in the dataset.")

# Assign Author and Body
df_raw['Author'] = df_raw[author_column]
df_raw['Body'] = df_raw[text_column]

# Clean out forwarded emails and replies using the Subject column
if 'Subject' in df_raw.columns:
    # Filter out subjects starting with RE:, FW:, FWD: (case insensitive)
    df_raw = df_raw[~df_raw['Subject'].astype(str).str.match(r'^\s*(re|fw|fwd)\s*:', case=False, na=False)]

df_raw = df_raw.dropna(subset=['Author', 'Body'])

# --- Better Text Filtering ---
def clean_enron_body(text):
    text = str(text)
    # Lowercase text (remove capitalization)
    text = text.lower()

    # Remove ONLY URLs using textacy
    text = preprocessing.replace.urls(text, repl="")

    # Remove 'Original Message' or 'Forwarded by' blocks and everything after
    text = re.sub(r'-+original message-+.*', '', text, flags=re.DOTALL)
    text = re.sub(r'-+forwarded by-+.*', '', text, flags=re.DOTALL)
    # Remove email metadata headers like To:, From:, Sent:, cc:, bcc:, Subject: if they appear in body
    text = re.sub(r'^(to|from|sent|cc|bcc|subject):\s.*$', '', text, flags=re.MULTILINE)

    # Remove multiple whitespaces/newlines
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_raw['Body'] = df_raw['Body'].apply(clean_enron_body)

# --- EXACT MIRROR OF R SCRIPT: split_data.R filter_authors_by_count ---
author_counts = df_raw['Author'].value_counts()
# Exactly 20 to 30 documents per author
valid_authors = author_counts[(author_counts >= 20) & (author_counts <= 30)].index
df = df_raw[df_raw['Author'].isin(valid_authors)].reset_index(drop=True)

print(f"Selected {len(df)} emails from {len(df['Author'].unique())} authors (strictly 20-30 docs per author).")


Selected 6739 emails from 274 authors (strictly 20-30 docs per author).


### 3. Extract Stylistic Features

In [49]:
def extract_stylistic_features(texts):
    features_list = []
    for doc in nlp.pipe(texts, batch_size=50):
        total_tokens = max(len(doc), 1) # Prevent division by zero
        pos_counts = Counter([token.pos_ for token in doc])

        # Extract raw counts and CTTR/AWL
        features = {
            'Tokens': total_tokens,
            'CTTR': len(set([t.text.lower() for t in doc])) / np.sqrt(2 * total_tokens),
            'AWL': np.mean([len(t.text) for t in doc if t.is_alpha]) if any(t.is_alpha for t in doc) else 0
        }

        # Add raw counts and relative frequencies for each POS tag
        pos_tags = ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SPACE', 'VERB']
        for pos in pos_tags:
            features[pos] = pos_counts.get(pos, 0)
            features[f'Rel{pos}'] = features[pos] / total_tokens

        features_list.append(features)
    return pd.DataFrame(features_list)

print("Extracting raw and relative linguistic features...")
df_features = extract_stylistic_features(df['Body'].astype(str).tolist())
df_combined = pd.concat([df.reset_index(drop=True), df_features], axis=1)

Extracting raw and relative linguistic features...


### 4. Create Interaction Terms & Split Data

In [57]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import random
import numpy as np
import math

# ==== 1. EXACT AUTHOR SPLIT FROM split_data.R ====
seed = 1923
np.random.seed(seed)
random.seed(seed)

authors = df_combined['Author'].unique()
shuffled_authors = list(authors)
random.shuffle(shuffled_authors)

# "total_size <- floor(0.95 * length(shuffled_authors))"
total_size = math.floor(0.95 * len(shuffled_authors))

# "train_size <- floor(split_ratio[1] * total_size)"
train_size = math.floor(0.7 * total_size)

# Modifying to exactly 5 authors in test set as requested
test_size = 5
valid_size = total_size - train_size - test_size

train_authors = shuffled_authors[:train_size]
valid_authors = shuffled_authors[train_size:train_size + valid_size]
test_authors = shuffled_authors[train_size + valid_size:train_size + valid_size + test_size]

train_df = df_combined[df_combined['Author'].isin(train_authors)].copy().reset_index(drop=True)
val_df = df_combined[df_combined['Author'].isin(valid_authors)].copy().reset_index(drop=True)
test_df = df_combined[df_combined['Author'].isin(test_authors)].copy().reset_index(drop=True)

print(f"Author Split: Train={len(train_authors)}, Valid={len(valid_authors)}, Test={len(test_authors)}")
print(f"Document Split: Train={len(train_df)}, Valid={len(val_df)}, Test={len(test_df)}")

# ==== 2. BUILD PROFILES USING RESPECTIVE CANDIDATE POOLS ====
pos_tags = ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SPACE', 'VERB']
stylistic_cols = [f'Rel{pos}' for pos in pos_tags] + ['CTTR', 'AWL']
interaction_features = ['RelADJ', 'RelDET', 'RelNOUN', 'RelPRON', 'RelPROPN', 'RelPUNCT', 'RelSPACE']

def build_author_profiles(data_df):
    profiles = data_df.groupby('Author')['Body'].apply(lambda x: ' '.join(x.astype(str))).to_dict()
    style_profiles = {}
    for author, group in data_df.groupby('Author'):
        author_tokens = max(group['Tokens'].sum(), 1)
        style = {}
        for pos in pos_tags:
            style[f'Rel{pos}'] = group[pos].sum() / author_tokens

        tokens = profiles[author].split()
        total_toks = max(len(tokens), 1)
        types = len(set(t.lower() for t in tokens))
        style['CTTR'] = types / np.sqrt(2 * total_toks)
        alpha_toks = [t for t in tokens if t.isalpha()]
        style['AWL'] = np.mean([len(t) for t in alpha_toks]) if alpha_toks else 0
        style_profiles[author] = style
    return profiles, style_profiles

train_text_profiles, train_style_profiles = build_author_profiles(train_df)
val_text_profiles, val_style_profiles = build_author_profiles(val_df)

# Vectorizer to compute ngram_sim later
tfidf_vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 3), max_features=10000)
tfidf_vectorizer.fit(list(train_text_profiles.values()))

# ==== 3. GENERATE PAIRWISE DATASET PER SPLIT ====
def create_pairwise_dataset(data_df, author_texts, author_styles):
    candidates_list = list(author_texts.keys())
    profile_texts_list = [author_texts[a] for a in candidates_list]
    profile_vectors = tfidf_vectorizer.transform(profile_texts_list)

    doc_texts = data_df['Body'].astype(str).tolist()
    doc_vectors = tfidf_vectorizer.transform(doc_texts)
    sim_matrix = cosine_similarity(doc_vectors, profile_vectors)

    records = []
    for i, row in enumerate(data_df.itertuples()):
        true_author = row.Author
        for author_idx, author in enumerate(candidates_list):
            ngram_sim = sim_matrix[i, author_idx]
            cand_style = author_styles[author]

            record = {}
            for col in stylistic_cols:
                record[col] = abs(getattr(row, col) - cand_style[col])
            record['ngram_sim'] = ngram_sim
            for feat in interaction_features:
                record[f'ngram_sim_x_{feat}'] = ngram_sim * record[feat]

            record['Label'] = 1 if author == true_author else 0
            record['Doc_ID'] = i
            record['Candidate'] = author
            record['True_Author'] = true_author
            records.append(record)
    return pd.DataFrame(records)

print("Generating pairwise features for Train and Valid sets...")
train_pairs = create_pairwise_dataset(train_df, train_text_profiles, train_style_profiles)
val_pairs = create_pairwise_dataset(val_df, val_text_profiles, val_style_profiles)

top_base_features = interaction_features + ['CTTR']
interaction_cols = [f'ngram_sim_x_{feat}' for feat in interaction_features]
feature_cols = ['ngram_sim'] + top_base_features + interaction_cols

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_pairs[feature_cols])
y_train = train_pairs['Label'].values

X_val_scaled = scaler.transform(val_pairs[feature_cols])
y_val = val_pairs['Label'].values


Author Split: Train=182, Valid=73, Test=5
Document Split: Train=4456, Valid=1816, Test=128
Generating pairwise features for Train and Valid sets...


### 5. Model Training (Elastic Net)

In [53]:
from sklearn.linear_model import LogisticRegression

# The original R code uses glmnet with alpha=0.8 (which maps to l1_ratio=0.8 in scikit-learn)
# and standardize=FALSE. We already scaled our data using StandardScaler earlier.

print("Training Binary Elastic Net classifier matching exact R configuration (l1_ratio=0.8, no class weights)...")

# Using SGDClassifier with log_loss and l1_ratio=0.8 to perfectly mirror glmnet's elasticnet
# Removed class_weight='balanced' to match R's default glmnet behavior
model = SGDClassifier(
    loss='log_loss',
    penalty='elasticnet',
    l1_ratio=0.8,
    max_iter=2000,
    random_state=42
)

# The R code uses cv.glmnet to find the best lambda (alpha in sklearn's SGD)
# cv.glmnet defaults to 10-fold CV
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
}

cv_size = min(len(X_train_scaled), 10000)
grid_search = GridSearchCV(model, param_grid, cv=10, scoring='roc_auc')
grid_search.fit(X_train_scaled[:cv_size], y_train[:cv_size])

best_model = grid_search.best_estimator_
print(f"Optimal lambda (alpha) found: {best_model.alpha}")

# Retrain on the full training set with the best penalty parameter
best_model.fit(X_train_scaled, y_train)


Training Binary Elastic Net classifier matching exact R configuration (l1_ratio=0.8, no class weights)...
Optimal lambda (alpha) found: 0.001


SGDClassifier(alpha=0.001, l1_ratio=0.8, loss='log_loss', max_iter=2000,
              penalty='elasticnet', random_state=42)

### 6. Final Evaluation

In [58]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

# 1. Build candidate profiles for the Test set authors
print(f"Evaluating {len(test_df)} documents from the test set...")

test_text_profiles, test_style_profiles = build_author_profiles(test_df)
test_candidates = list(test_text_profiles.keys())

print(f"Evaluating against the {len(test_candidates)} test candidate authors...")

test_records = []

# 2. Get test document texts and vectors
doc_texts = test_df['Body'].astype(str).tolist()
doc_vectors = tfidf_vectorizer.transform(doc_texts)

# 3. Candidate author profiles
candidate_profile_texts = [test_text_profiles[a] for a in test_candidates]
profile_vectors = tfidf_vectorizer.transform(candidate_profile_texts)

# Compute similarity matrix between test docs and test candidate profiles
sim_matrix = cosine_similarity(doc_vectors, profile_vectors)

for i, row in enumerate(test_df.itertuples()):
    true_author = row.Author

    for author_idx, author in enumerate(test_candidates):
        ngram_sim = sim_matrix[i, author_idx]
        cand_style = test_style_profiles[author]

        record = {}
        # Document - Author differences
        for col in stylistic_cols:
            doc_val = getattr(row, col)
            cand_val = cand_style[col]
            record[col] = abs(doc_val - cand_val)

        record['ngram_sim'] = ngram_sim

        # Restrict interactions to paper's set
        for feat in interaction_features:
            if feat in record:
                record[f'ngram_sim_x_{feat}'] = ngram_sim * record[feat]

        record['Label'] = 1 if author == true_author else 0
        record['Doc_ID'] = i
        record['Candidate'] = author
        record['True_Author'] = true_author
        test_records.append(record)

test_pairs = pd.DataFrame(test_records)
X_test_scaled_final = scaler.transform(test_pairs[feature_cols])

# Predict probabilities for each test candidate
test_pairs['Probability'] = best_model.predict_proba(X_test_scaled_final)[:, 1]

predictions = []
truths = []

for doc_id, group in test_pairs.groupby('Doc_ID'):
    # Top-1 selection
    best_candidate = group.loc[group['Probability'].idxmax()]['Candidate']
    true_author = group.iloc[0]['True_Author']

    predictions.append(best_candidate)
    truths.append(true_author)

print(f"\n--- Final Model Evaluation on Test Set ---")
print(f"Top-1 Authorship Attribution Accuracy: {accuracy_score(truths, predictions):.4f}")


Evaluating 128 documents from the test set...
Evaluating against the 5 test candidate authors...

--- Final Model Evaluation on Test Set ---
Top-1 Authorship Attribution Accuracy: 0.9141
